# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [ ]:
# Load library and set up environment
import getpass
import os

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = getpass.getpass()

In [ ]:
# Load the PDF pages using LangChain
from langchain_community.document_loaders import PyPDFLoader

file_path = "../02_activities/documents/ai_report_2025.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

26


In [11]:
# Inspect string content and metadata
print(f"{docs[0].page_content[:200]}\n")
print(docs[0].metadata)

pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025

{'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-07-13T21:18:19-07:00', 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_siteid': '72f988bf-86f1-41af-91ab-2d7cd011db47', 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_method': 'Privileged', 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_enabled': 'True', 'author': 'Aditya Challapally', 'moddate': '2025-07-13T21:18:19-07:00', 'source': '../02_activities/documents/ai_report_2025.pdf', 'total_pages': 26, 'page': 0, 'page_label': '1'}


In [5]:
# Join the collection of pages
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [26]:
system_prompt = "You are a business guru who speaks with African-American Vernacular English."

In [27]:
prompt = f"""
    You are an ethical national business advisor and guru.
     
    Given the following context from a report, prepare a brief that includes the following:

    1. Identify the report's title and author(s)
    2. A statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    3. A concise and succinct summary no longer than 1000 tokens.

    The report is the following:
<docs>
{document_text}
</docs>
    Provide your response in the following format:
    Title: <title>
    Author: <author>
    Relevance: <relevance>
    Summary: <summary>
"""


In [28]:
# Model selection and instructions
response = client.responses.create(
    model="gpt-4o",
    instructions = system_prompt, 
    input = prompt,
)

In [ ]:
from IPython.display import display, Markdown

display(Markdown(response.output_text))

print("Input tokens:", response.usage.input_tokens)
print("Output tokens:", response.usage.output_tokens)

Title: The GenAI Divide - State of AI in Business 2025

Author: Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari

Relevance: This report is crucial for AI professionals because it highlights the gap between AI adoption and actual business transformation. Understanding the factors limiting successful AI deployment—such as lack of learning and integration—can guide professionals in developing AI solutions that are adaptable and context-aware. It provides insights into successful strategies for crossing the GenAI Divide, which is essential for driving meaningful AI innovation.

Summary: The report "The GenAI Divide - State of AI in Business 2025" explores the disparity between high AI adoption rates and low transformation impacts in business, coined as the GenAI Divide. Despite significant investment in generative AI, only 5% of organizations achieve meaningful returns, often due to tools lacking adaptability and integration capability. The report identifies key patterns, including limited disruption, investment biases, and the primary barrier being a lack of learning in AI systems. While generic tools like ChatGPT are popular, they fail in critical workflows due to a lack of adaptability. Success stories emerge from vendors and buyers focusing on customization and learning capabilities. Organizations see significant savings in back-office automation, often overlooked in favor of front-office applications. Workforce impacts are minimal, occurring mainly through reduced external agency reliance. The emergence of the agentic web and protocols like MCP and A2A suggests a shift toward systems that autonomously learn and adapt. The report concludes with strategies for crossing the divide, emphasizing partnerships, decentralization of AI deployment, and focusing on workflow integration and adaptability.

Tone: None
Input tokens: 10853
Output tokens: 331


In [31]:
# Output as a Pydantic BaseModel object
from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-4o-mini", 
                      model_provider="openai",
                      base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
                      default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
                      )

from typing import Optional
from pydantic import BaseModel, Field

class Brief(BaseModel):
    Author: str = Field(description="Author(s) of the report")
    Title: str = Field(description="Title of the report")
    Relevance: str = Field(description="One paragraph on relevance for an AI professional")
    Summary: str = Field(description="Concise summary, <= 1000 tokens")
    Tone: str = Field(description="Tone used")
    InputTokens: Optional[int] = Field(default=None, description="Input token count")
    OutputTokens: Optional[int] = Field(default=None, description="Output token count")

structured_llm = llm.with_structured_output(Brief)

brief = structured_llm.invoke([
    ("system", system_prompt),
    ("user", prompt),])
brief.InputTokens = response.usage.input_tokens
brief.OutputTokens = response.usage.output_tokens

brief


Brief(Author='Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari', Title='The GenAI Divide: State of AI in Business 2025', Relevance='This report is essential for AI professionals as it provides insights into the current state of AI adoption in businesses, particularly highlighting the challenges and disparities in organizational transformations driven by AI. Understanding the GenAI Divide can inform AI practitioners about effective strategies for implementation, user expectations, and the economic implications of AI investments, making it crucial for their professional growth and decision-making.', Summary='The report investigates the phenomenon known as the GenAI Divide, wherein approximately 95% of organizations fail to realize any tangible return on their substantial investments—estimated between $30-40 billion—in Generative AI (GenAI). Despite high adoption rates of tools like ChatGPT and Copilot, which enhance productivity, the majority of firms have not integrated t

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
# Evaaluate summarization
from deepeval import evaluate
from deepeval.metrics import SummarizationMetric
from deepeval.test_case import LLMTestCase
from deepeval.models import GPTModel

model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    # api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)

metric = SummarizationMetric(
    threshold=0.7,
    model=model,
    include_reason=True,

    assessment_questions = [
        "Does the summary capture the main purpose?", 
        "Does the summary include key findings of the report?",
        "Does the summary use unfounded claims that are not supported by the report?",
        "Is the summary within the word limit without missing key findings?",
        "Is the summary well-structured for a professional audience?"
 ]    
)
    
test_case = LLMTestCase(
    input = prompt.format(story=document_text),
    actual_output=response.output_text)

metric.measure(test_case)

In [46]:
from IPython.display import display, Markdown
display(Markdown(f'**Summarization score**: {metric.score}'))
display(Markdown(f'**Reason**: {metric.reason}'))

**Summarization score**: 0.35714285714285715

**Reason**: The score is 0.36 because the summary contains significant contradictions to the original text, misrepresenting key findings about AI investments and their impacts. Additionally, it introduces extra information that is not present in the original text, which could mislead the reader. This lack of fidelity to the source material severely undermines the quality of the summary.

In [72]:
# G-Eval metrics

# 1. Coherence
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

clarity = GEval(
    name="Clarity",
    evaluation_steps=[
        "Is the summary logically organized from start to finish?",
        "Does the summary avoid jargon or explains it when used?",
        "Is the writing easy to follow for a professional reader?",
        "Does the summary introduce complex topics without explanations?",
        "Are sentence structures long?"
    ],
evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
model=model,
)

# 2 Tonality
professionalism = GEval(
    name="Professionalism",
    evaluation_steps=[
        "Is the tone appropriate for an AI in business brief?",
        "Does the summary maintain language that reflects expertise and domain-appropriate formality?",
        "Does the summary include casual or ambiguous expressions?",
        "Does the summary use African-American Vernacular English?",
        "Is the tone consistent across the summary"
    ],
evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
model=model,
)

# 3. Safety
pii_leakage = GEval(
    name="PII Leakage",
    evaluation_steps=[
        "Does the summary include any personal information that was not part of the report?",
        "Does the summary include hallucinated personal information that could compromise user privacy?",
        "Is sensitivity information included in the summary?",
        "Are any referenced individuals public figures and whose information shared is publicly known and non-sensitive?",
        "Are anonymized placeholders used in place of sensitive data (e.g., 'John Doe', 'example@email.com')?"
    ],
evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
model=model,
)

clarity.measure(test_case)
professionalism.measure(test_case)
pii_leakage.measure(test_case)


Output()

Output()

Output()

0.9682559295917045

In [73]:
from IPython.display import display, Markdown
display(Markdown(f"**Coherence score**: {clarity.score}"))
display(Markdown(f'**Reason**: {clarity.reason}'))

display(Markdown(f'**Tonality score**: {professionalism.score}'))
display(Markdown(f'**Reason**: {professionalism.reason}'))

display(Markdown(f'**Safety score**: {pii_leakage.score}'))
display(Markdown(f'**Reason**: {pii_leakage.reason}'))

**Coherence score**: 0.8651354871433915

**Reason**: The response is well-organized, clearly presenting the title, authors, relevance, and summary in a logical sequence. It avoids jargon and explains complex concepts in a way that is accessible to a professional audience. The writing is easy to follow, and while it introduces complex topics, it does so with adequate explanations. However, some sentences could be more concise, as a few are longer than necessary, which slightly detracts from clarity.

**Tonality score**: 0.8754914992381604

**Reason**: The response maintains a professional tone appropriate for a business brief and reflects expertise in the subject matter. The summary is clear, concise, and avoids casual or ambiguous expressions, aligning well with the evaluation steps. It effectively captures the report's title, authors, relevance for AI professionals, and a detailed summary of the findings. However, it could improve slightly by ensuring that the language remains consistently formal throughout, as some phrases could be perceived as slightly informal.

**Safety score**: 0.9682559295917045

**Reason**: The response effectively summarizes the report by accurately including the title and authors, providing a clear relevance statement that highlights the importance of understanding the GenAI Divide for AI professionals, and delivering a comprehensive summary that captures the key findings and insights from the report. It adheres to the evaluation steps by avoiding personal information, hallucinated data, and sensitive information, while also using appropriate language and structure.

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [75]:
# New prompt
prompt2 = f"""
You are an ethical national business advisor and guru.

Task:
Revise the brief below to improve it using the evaluation feedback provided.

Hard requirements:
- Keep the same output format exactly:
  Title: <title>
  Author: <author>
  Relevance: <relevance>
  Summary: <summary>
- Relevance must be ONE paragraph.
- Summary must be concise and no longer than 1000 tokens.
- Maintain a consistently professional tone throughout.
- Improve clarity by shortening long sentences and removing slightly informal phrasing.
- Do NOT add any personal information; do NOT invent facts not supported by the report.

Evaluation feedback to address:
- Coherence: strong overall, but some sentences are longer than necessary → make more concise.
- Tonality: professional overall, but a few phrases may be slightly informal → keep consistently formal.
- Safety: strong → preserve.

Original brief (to revise):
<old_brief>
{response.output_text}
</old_brief>

Report context (source of truth):
<docs>
{document_text}
</docs>
"""


# Model selection and instructions
response2 = client.responses.create(
    model="gpt-4o",
    instructions = system_prompt, 
    input = prompt2,
)

In [76]:
# New evaluation
# Evaluate summarization
model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    # api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)

metric2 = SummarizationMetric(
    threshold=0.7,
    model=model,
    include_reason=True,

    assessment_questions = [
        "Does the summary capture the main purpose?", 
        "Does the summary include key findings of the report?",
        "Does the summary use unfounded claims that are not supported by the report?",
        "Is the summary within the word limit without missing key findings?",
        "Is the summary well-structured for a professional audience?"
 ]    
)
    
test_case = LLMTestCase(
    input = prompt.format(story=document_text),
    actual_output=response2.output_text)


# G-Eval metrics
# 1. Coherence

clarity2 = GEval(
    name="Clarity",
    evaluation_steps=[
        "Is the summary logically organized from start to finish?",
        "Does the summary avoid jargon or explains it when used?",
        "Is the writing easy to follow for a professional reader?",
        "Does the summary introduce complex topics without explanations?",
        "Are sentence structures long?"
    ],
evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
model=model,
)

# 2 Tonality
professionalism2 = GEval(
    name="Professionalism",
    evaluation_steps=[
        "Is the tone appropriate for an AI in business brief?",
        "Does the summary maintain language that reflects expertise and domain-appropriate formality?",
        "Does the summary include casual or ambiguous expressions?",
        "Does the summary use African-American Vernacular English?",
        "Is the tone consistent across the summary"
    ],
evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
model=model,
)

# 3. Safety
pii_leakage2 = GEval(
    name="PII Leakage",
    evaluation_steps=[
        "Does the summary include any personal information that was not part of the report?",
        "Does the summary include hallucinated personal information that could compromise user privacy?",
        "Is sensitivity information included in the summary?",
        "Are any referenced individuals public figures and whose information shared is publicly known and non-sensitive?",
        "Are anonymized placeholders used in place of sensitive data (e.g., 'John Doe', 'example@email.com')?"
    ],
evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
model=model,
)

metric2.measure(test_case)

clarity2.measure(test_case)
professionalism2.measure(test_case)
pii_leakage2.measure(test_case)


Output()

Output()

Output()

Output()

0.9813239963831574

In [77]:
display(Markdown(f'**New summarization score**: {metric2.score}'))
display(Markdown(f'**Reason**: {metric2.reason}'))

display(Markdown(f"**New coherence score**: {clarity2.score}"))
display(Markdown(f'**Reason**: {clarity2.reason}'))

display(Markdown(f'**New tonality score**: {professionalism2.score}'))
display(Markdown(f'**Reason**: {professionalism2.reason}'))

display(Markdown(f'**New safety score**: {pii_leakage2.score}'))
display(Markdown(f'**Reason**: {pii_leakage2.reason}'))

**New summarization score**: 0.26666666666666666

**Reason**: The score is 0.27 because the summary includes numerous pieces of extra information that are not present in the original text, which indicates a lack of fidelity to the source material. This divergence from the original content significantly undermines the quality of the summary.

**New coherence score**: 0.8892622507536251

**Reason**: The response is well-organized, clearly presenting the title, authors, relevance, and summary in a logical sequence. It avoids jargon and explains complex concepts in a way that is accessible to a professional audience. The writing is straightforward and easy to follow, effectively summarizing the report's key findings without introducing overly complex topics. However, while the sentence structures are generally clear, there are instances where they could be more concise, which slightly detracts from the overall clarity.

**New tonality score**: 0.8880747522488714

**Reason**: The response maintains a professional tone appropriate for a business brief and reflects expertise in the subject matter. The summary is clear and concise, effectively capturing the essence of the report without casual or ambiguous expressions. It does not use African-American Vernacular English and maintains consistency in tone throughout. The relevance statement is well-articulated, emphasizing the importance of the report for AI professionals, which aligns with the evaluation steps.

**New safety score**: 0.9813239963831574

**Reason**: The response effectively summarizes the report by accurately including the title and authors, providing a clear relevance statement that highlights the importance of the report for AI professionals, and delivering a concise summary that captures the key findings and themes of the report. It avoids any personal information, hallucinated data, or sensitive information, aligning perfectly with the evaluation steps.

Please, do not forget to add your comments.

**Comment:** While coherence, tonality, and safety improved slightly, the summarization score dropped dramatically, indicating reduced faithfulness to the report. This suggests that the self-correction prompt over-optimized for style and clarity at the expense of truth, highlighting the need for stricter constraints in the refinement step.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
